In [16]:
import pandas as pd
import json
import sys
import os

# Import section that will work everywhere I want it to
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    print('Seems like this notebook runs in Kaggle, we will not import anything additional')
else:
    try:
        from google.colab import userdata
        from google.colab import drive
        drive.mount('/content/drive')
        print('Seems like this notebook runs in Google Colab. If not - please check import sequence and change a code')
    except:
        from dotenv import load_dotenv
        load_dotenv()
        print('Seems like this notebook runs in local environment, loaded .env file')

Seems like this notebook runs in Kaggle, we will not import anything


In [18]:
# Getting file path depending on an environment
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print("This notebook is running on Kaggle. We are using file path for Kaggle")
    file_path = '/kaggle/input/datasets/nikolaybaakh/nasa-data-on-all-asteroids-neows/all_neos.csv'
elif 'google.colab' in sys.modules:
    print("Running in Google Colab")
    file_path = '/content/drive/MyDrive/temp_colab_data/all_neos.csv'
else:
    print("This notebook is running outside of Kaggle (e.g., locally or on another platform). We are using file path for local work")
    file_path = '../data/nasa_data/all_neos.csv'

This notebook is running on Kaggle. We are using file path for Kaggle


# Data Preparation

In [8]:
all_neos_df = pd.read_csv(file_path)
all_neos_df.head()

,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid,close_approach_data,orbital_data,is_sentry_object,page
0,2000433,2000433,433 Eros (A898 PA),433,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,10.38,{'kilometers': {'estimated_diameter_min': 22.3...,False,"[{'close_approach_date': '1900-12-27', 'close_...","{'orbit_id': '659', 'orbit_determination_date'...",False,0
1,2000719,2000719,719 Albert (A911 TB),719,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,15.59,{'kilometers': {'estimated_diameter_min': 2.02...,False,"[{'close_approach_date': '1909-08-21', 'close_...","{'orbit_id': '274', 'orbit_determination_date'...",False,0
2,2000887,2000887,887 Alinda (A918 AA),887,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.81,{'kilometers': {'estimated_diameter_min': 4.59...,False,"[{'close_approach_date': '1974-01-04', 'close_...","{'orbit_id': '730', 'orbit_determination_date'...",False,0
3,2001036,2001036,1036 Ganymed (A924 UB),1036,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,9.18,{'kilometers': {'estimated_diameter_min': 38.7...,False,"[{'close_approach_date': '1910-02-25', 'close_...","{'orbit_id': '1460', 'orbit_determination_date...",False,0
4,2001221,2001221,1221 Amor (1932 EA1),1221,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,17.37,{'kilometers': {'estimated_diameter_min': 0.89...,False,"[{'close_approach_date': '1908-03-14', 'close_...","{'orbit_id': '143', 'orbit_determination_date'...",False,0


In [9]:
json.loads(all_neos_df.estimated_diameter[0].replace("'", '"'))['kilometers']['estimated_diameter_max']

49.8930414151

In [10]:
all_neos_df.estimated_diameter[-all_neos_df.estimated_diameter.apply(lambda x: isinstance(x,str))]

7113     NaN
7300     NaN
7312     NaN
7315     NaN
7416     NaN
7419     NaN
7420     NaN
24926    NaN
28983    NaN
29120    NaN
29121    NaN
29363    NaN
38501    NaN
Name: estimated_diameter, dtype: object

That's sad, but I don't think that this is important

In [ ]:
all_neos_df.estimated_diameter.apply(lambda x: json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_max'] if isinstance(x, str) else None)

0        49.893041
1         4.529393
2        10.281109
3        86.704169
4         1.995446
           ...    
42101     0.003369
42102     0.034454
42103     0.073661
42104     0.167201
42105     0.023038
Name: estimated_diameter, Length: 42106, dtype: float64

In [132]:
# Converting estimated diameter to meters and getting it from json
all_neos_df['estimated_diameter_meters_max'] = all_neos_df.estimated_diameter.apply(lambda x: round(json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_max'] * 1000,2) if isinstance(x, str) else None)
all_neos_df['estimated_diameter_meters_min'] = all_neos_df.estimated_diameter.apply(lambda x: round(json.loads(x.replace("'", '"'))['kilometers']['estimated_diameter_min'] * 1000,2) if isinstance(x, str) else None)
all_neos_df.drop('estimated_diameter', axis = 1, inplace=True) # dropping json as we got all we needed

all_neos_df.head()

,id,neo_reference_id,name,designation,nasa_jpl_url,absolute_magnitude_h,is_potentially_hazardous_asteroid,close_approach_data,orbital_data,is_sentry_object,page,estimated_diameter_meters_max,estimated_diameter_meters_min
0,2000433,2000433,433 Eros (A898 PA),433,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,10.38,False,"[{'close_approach_date': '1900-12-27', 'close_...","{'orbit_id': '659', 'orbit_determination_date'...",False,0,49893.04,22312.85
1,2000719,2000719,719 Albert (A911 TB),719,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,15.59,False,"[{'close_approach_date': '1909-08-21', 'close_...","{'orbit_id': '274', 'orbit_determination_date'...",False,0,4529.39,2025.61
2,2000887,2000887,887 Alinda (A918 AA),887,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,13.81,False,"[{'close_approach_date': '1974-01-04', 'close_...","{'orbit_id': '730', 'orbit_determination_date'...",False,0,10281.11,4597.85
3,2001036,2001036,1036 Ganymed (A924 UB),1036,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,9.18,False,"[{'close_approach_date': '1910-02-25', 'close_...","{'orbit_id': '1460', 'orbit_determination_date...",False,0,86704.17,38775.28
4,2001221,2001221,1221 Amor (1932 EA1),1221,https://ssd.jpl.nasa.gov/tools/sbdb_lookup.htm...,17.37,False,"[{'close_approach_date': '1908-03-14', 'close_...","{'orbit_id': '143', 'orbit_determination_date'...",False,0,1995.45,892.39


In [133]:
# Creating orbital data as a separate DataFrame. We will not parse it since we don't know if we will need it
orbital_data = all_neos_df[['id', 'neo_reference_id', 'name', 'orbital_data']].copy()
orbital_data.head()

,id,neo_reference_id,name,orbital_data
0,2000433,2000433,433 Eros (A898 PA),"{'orbit_id': '659', 'orbit_determination_date'..."
1,2000719,2000719,719 Albert (A911 TB),"{'orbit_id': '274', 'orbit_determination_date'..."
2,2000887,2000887,887 Alinda (A918 AA),"{'orbit_id': '730', 'orbit_determination_date'..."
3,2001036,2001036,1036 Ganymed (A924 UB),"{'orbit_id': '1460', 'orbit_determination_date..."
4,2001221,2001221,1221 Amor (1932 EA1),"{'orbit_id': '143', 'orbit_determination_date'..."


In [ ]:
# Creating close approach data as a separate DataFrame
close_approach_data = pd.DataFrame()

for i in range(len(all_neos_df)):
    temp_df = pd.DataFrame(json.loads(all_neos_df.close_approach_data[i].replace("'", '"')))
    temp_df['neo_reference_id'] = all_neos_df.neo_reference_id[i]
    temp_df['id'] = all_neos_df.id[i]
    temp_df['name'] = all_neos_df.name[i]
    temp_df['absolute_magnitude_h'] = all_neos_df.absolute_magnitude_h[i]
    temp_df['is_sentry_object'] = all_neos_df.is_sentry_object[i]
    temp_df['estimated_diameter_meters_max'] = all_neos_df.estimated_diameter_meters_max[i]
    temp_df['is_potentially_hazardous_asteroid'] = all_neos_df.is_potentially_hazardous_asteroid[i]
    try:
        temp_df['relative_velocity_kph'] = temp_df.relative_velocity.apply(lambda x: x['kilometers_per_hour'])
        temp_df['miss_distance_meters'] = temp_df.miss_distance.apply(lambda x: x['kilometers'] * 1000)
        close_approach_data = pd.concat([close_approach_data, temp_df])
    except:
        pass

close_approach_data.drop(['relative_velocity', 'miss_distance'], axis=1, inplace=True)
close_approach_data.reset_index(inplace=True,drop=True)
close_approach_data.head()

# Research

What interesting could I find on NASA's Data?

We will distribute our analysis on datasets. First dataset is asteroids themselves:
1. How much asteroids are we observe?
2. Median Magnitude of asteroids.
3. How big are asteroids on average compared to real stuff?
4. What is the biggest asteroid? How big is it?
5. How many asteroids potentially hazardous? What's the difference between hazardous and non-hazardous?
6. How much Sentry asteroids are there? And what the hell are they?

Next we will go through close_approach_data:
0. How much asteroids orbit non earth flying around us?
1. When amount of asteroids close to earth will or were the highest?
2. When amount of hazardous asteroids close to earth will or were maximum?
3. When will we see the scariest asteroid? When have we seen it?
4. What's the closest asteroids? When were they closest?
5. When did we see fastest asteroids?

I will not research orbits, it will need too much knowledge